# Notebook: OBIS downloader (detailed)
What this notebook does:
- Orchestrates OBIS species downloads with resume support, storing chunked Parquet files per species.

Main steps:
1. Build a live species inventory via OBIS API (`fetch_live_inventory`).
2. For each species, run `download_obis_species` (Ray remote) which pages the occurrence API and writes chunked Parquet.
3. Use `get_resume_state` to resume partial downloads without re-reading whole files.

Key variables:
- `storage_dir`, `groups` for `fetch_live_inventory`, `chunk_size` inside the worker.

Runtime notes:
- Network reliability matters; the worker retries on network errors. Expect many small Parquet files per species used for incremental resume.

# OBIS downloader orchestration
This notebook collects live OBIS inventories and runs distributed downloads for species occurrences, writing chunked Parquet files per species.

- Purpose: create a resumable dataset of OBIS occurrences organized by higher-level category and species.
- Usage: inspect `fetch_live_inventory()` and `download_obis_species()` functions; set `storage_dir` and start Ray before launching orchestrator tasks.
- Notes: The notebook uses `pyarrow` metadata reads to resume downloads without reloading full data into RAM.

In [1]:
import ray
import requests
import pandas as pd
import pyarrow.parquet as pq
import os
import glob
import time

### ==========================================
### 1. Inventory Query
### ==========================================

In [ ]:
def fetch_live_inventory(groups=["Elasmobranchii", "Holocephali", "Scombridae", "Istiophoridae", "Xiphiidae", "Clupeidae", "Gadidae", "Pleuronectiformes", "Otariidae", "Odobenidae", "Sirenia", "Dermochelyidae", "Sphenisciformes","Cephalopoda"], start_year=2000):
    """Holt das aktuellste Inventar direkt von OBIS, bevor der Download startet."""
    print("Erstelle Live-Inventar von OBIS...")
    all_dfs = []
    
    for group in groups:
        url = "https://api.obis.o   rg/v3/facet"
        params = {"facets": "scientificName", "scientificname": group, "startdate": f"{start_year}-01-01"}
        
        try:
            res = requests.get(url, params=params)
            if res.status_code == 200:
                results = res.json().get("results", {}).get("scientificName", [])
                if results:
                    df = pd.DataFrame(results).rename(columns={"key": "scientificName", "records": "count"})
                    df["category"] = group # Hier speichern wir die Überart
                    all_dfs.append(df)
                    print(f"  -> {group}: {len(df)} Spezies gefunden.")
        except Exception as e:
            print(f"  -> Fehler bei {group}: {e}")
            
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame()

### ==========================================
### 2. Helper For Download State
### ==========================================

In [3]:
def get_resume_state(folder_path):
    """
    Sucht die letzte ID und zählt alle bisher geladenen Zeilen.
    Nutzt pyarrow für extrem schnelles Auslesen der Metadaten (ohne RAM-Belastung).
    """
    files = glob.glob(f"{folder_path}/*.parquet")
    if not files:
        return None, 0
    
    total_rows = 0
    for f in files:
        try:
            # Schnelles Lesen der Metadaten, ohne die ganze Datei in den RAM zu laden
            total_rows += pq.read_metadata(f).num_rows
        except Exception:
            pass
            
    # Neueste Datei für die last_id finden
    latest_file = max(files, key=os.path.getctime)
    try:
        df_latest = pd.read_parquet(latest_file, columns=['id'])
        if not df_latest.empty:
            last_id = df_latest.iloc[-1]['id']
            return last_id, total_rows
    except Exception as e:
        print(f"Warnung beim Lesen von {latest_file}: {e}")
        
    return None, total_rows

### ==========================================
### 3. RAY WORKER-TASK
### ==========================================

In [4]:
@ray.remote(num_cpus=1)
def download_obis_species(category, scientific_name, expected_count, storage_dir, start_year=2000):
    
    # NEU: Verschachtelte Ordnerstruktur (Überart -> Art)
    species_dir = os.path.join(storage_dir, category, scientific_name.replace(" ", "_").lower())
    os.makedirs(species_dir, exist_ok=True)
    
    start_date = f"{start_year}-01-01"
    chunk_size = 10000
    
    # NEU: Korrekte Wiederaufnahme des Counters
    last_id, all_downloaded = get_resume_state(species_dir)
    
    # Wenn wir schon alles haben, direkt beenden
    if all_downloaded >= expected_count and last_id:
        return f"ÜBERSPRUNGEN: {scientific_name} (Bereits {all_downloaded} Punkte vorhanden)"
    
    status_msg = f"RESUME bei ID {last_id} ({all_downloaded} bereits da)" if last_id else "NEUER DOWNLOAD"
    print(f"[START] {scientific_name} | Ziel: {expected_count} | {status_msg}")

    while True:
        url = f"https://api.obis.org/v3/occurrence?scientificname={scientific_name}&startdate={start_date}&size={chunk_size}"
        if last_id:
            url += f"&after={last_id}"
            
        try:
            r = requests.get(url, timeout=120)
            if r.status_code != 200:
                time.sleep(5) 
                continue
                
            results = r.json().get('results', [])
            if not results:
                break # Keine Daten mehr verfügbar
                
            df_chunk = pd.DataFrame(results)
            if 'eventDate' in df_chunk.columns:
                df_chunk['eventDate'] = pd.to_datetime(df_chunk['eventDate'], errors='coerce', utc=True)
                
            # Speichern
            timestamp = int(time.time() * 1000)
            chunk_file = os.path.join(species_dir, f"chunk_{timestamp}.parquet")
            df_chunk.to_parquet(chunk_file, index=False)
            
            # Counter und ID aktualisieren
            last_id = results[-1].get('id')
            all_downloaded += len(results) # Jetzt stimmt die Zahl auch nach einem Neustart!
            
            print(f"[{scientific_name}] {all_downloaded} / {expected_count} geladen...")
            time.sleep(1)
            
        except Exception as e:
            print(f"[NETZWERK-FEHLER] {scientific_name}: {e}. Retry...")
            time.sleep(5)
            
    return f"ERFOLG: {scientific_name} ({all_downloaded} Datensätze gesichert im Ordner {category})"

### ==========================================
### 4. ORCHESTRATOR
### ==========================================

In [5]:
# 1. Live-Inventar erstellen
inventory_df = fetch_live_inventory()
if inventory_df.empty:
    print("Konnte kein Inventar erstellen. Beende Programm.")
    exit(1)
    
# Optional: Inventar zur Sicherheit lokal als Backup speichern
inventory_df.to_csv("finflow_live_inventory.csv", index=False)

# Nur Arten mit mehr als 100 Einträgen berücksichtigen
active_tasks = inventory_df[inventory_df['count'] > 10]

# 2. Ray initialisieren
if not ray.is_initialized():
    ray.init(address='auto') # oder ray.init() für lokale Tests

CLUSTER_STORAGE_DIR = "/mnt/shared_data/finflow/obis_raw" 
print(f"\nStarte verteilten Download für {len(active_tasks)} Spezies in {CLUSTER_STORAGE_DIR}...")

# 3. Jobs verteilen
task_references = []
for index, row in active_tasks.iterrows():
    task_ref = download_obis_species.remote(
        category=row['category'],           # Überart (Cetacea, etc.)
        scientific_name=row['scientificName'], # Art (Megaptera novaeangliae, etc.)
        expected_count=row['count'],
        storage_dir=CLUSTER_STORAGE_DIR
    )
    task_references.append(task_ref)
    
# 4. Warten auf Abschluss
results = ray.get(task_references)

print("\nALLE DOWNLOADS ABGESCHLOSSEN!")
for res in results:
    print(res)
    
ray.shutdown()

Erstelle Live-Inventar von OBIS...
  -> Elasmobranchii: 10 Spezies gefunden.
  -> Holocephali: 10 Spezies gefunden.
  -> Scombridae: 10 Spezies gefunden.
  -> Istiophoridae: 10 Spezies gefunden.
  -> Xiphiidae: 2 Spezies gefunden.
  -> Clupeidae: 10 Spezies gefunden.
  -> Gadidae: 10 Spezies gefunden.
  -> Pleuronectiformes: 10 Spezies gefunden.
  -> Otariidae: 10 Spezies gefunden.
  -> Odobenidae: 2 Spezies gefunden.
  -> Sirenia: 4 Spezies gefunden.
  -> Dermochelyidae: 1 Spezies gefunden.
  -> Sphenisciformes: 10 Spezies gefunden.


2026-03-27 12:59:24,660	INFO worker.py:1669 -- Using address ray://10.10.1.98:10001 set in the environment variable RAY_ADDRESS
2026-03-27 12:59:24,732	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver


  -> Cephalopoda: 10 Spezies gefunden.


SIGTERM handler is not set because current thread is not the main thread.



Starte verteilten Download für 109 Spezies in /mnt/shared_data/finflow/obis_raw...
(download_obis_species pid=213702, ip=10.10.4.23) [START] Dermochelys coriacea | Ziel: 17657 | RESUME bei ID ffef1a41-607e-4f79-a1d9-3e08b8c44425 (17596 bereits da)
(download_obis_species pid=219682, ip=10.10.3.63) [START] Thunnus thynnus | Ziel: 5161 | RESUME bei ID ffcd5297-60c3-4145-8922-3d17a2dc04cf (5159 bereits da)

ALLE DOWNLOADS ABGESCHLOSSEN!
ÜBERSPRUNGEN: Scyliorhinus canicula (Bereits 236523 Punkte vorhanden)
ÜBERSPRUNGEN: Carcharhinus leucas (Bereits 233198 Punkte vorhanden)
ÜBERSPRUNGEN: Carcharhinus melanopterus (Bereits 223553 Punkte vorhanden)
ÜBERSPRUNGEN: Carcharhinus amblyrhynchos (Bereits 218657 Punkte vorhanden)
ÜBERSPRUNGEN: Squalus acanthias (Bereits 176266 Punkte vorhanden)
ÜBERSPRUNGEN: Bathyraja brachyurops (Bereits 156565 Punkte vorhanden)
ÜBERSPRUNGEN: Bathyraja albomaculata (Bereits 108270 Punkte vorhanden)
ÜBERSPRUNGEN: Squalus suckleyi (Bereits 104933 Punkte vorhanden)
ÜBE

In [24]:
if ray.is_initialized():
    ray.shutdown()